# Target Detection via Denoising Score Matching

Reproduces every experiment in the paper on the Pavia-University scene:

1. **IID single-class** background (Fig. 2) — AMF, GMM-Levin, L-DART, **DART**, L-LRao, LRao
2. **IID multi-class** background (Fig. 3)
3. **Spatial / correlated** background (Table 1) — adds AMF-local, DART-CFAR, **DARTS**, DARTS-CFAR

Runs on Google Colab or locally. Edit the **Algorithm hyperparameters** cell to
play with detector behaviour. Set `QUICK = True` for a fast smoke test.

## Setup (Colab or local)

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules

# This project is self-contained: the dataset (data/pavia-u.mat) is committed,
# so once the files are present nothing else needs downloading.
# To fetch it into Colab, set GIT_URL below and run this cell.
GIT_URL    = ''                  # e.g. 'https://github.com/<user>/<repo>.git'
GIT_BRANCH = 'reviewer-release'  # the branch that holds this clean release

def have_repo():
    return os.path.isfile('src/iid.py') and os.path.isfile('data/pavia-u.mat')

if not have_repo():
    if GIT_URL:
        # clone the RELEASE branch (the repo's default branch may be a different one)
        subprocess.run(['git', 'clone', '--depth', '1', '-b', GIT_BRANCH, GIT_URL, 'repo'], check=True)
        os.chdir('repo')
    for cand in ('SDSM', 'repo', 'final-paper-experiment'):
        if not have_repo() and os.path.isdir(cand):
            os.chdir(cand)
assert have_repo(), (
    'Project files not found. Set GIT_URL above (clone uses branch '
    f'{GIT_BRANCH!r}), or upload the project folder/.zip in Colab, then re-run.')

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
sys.path.insert(0, os.getcwd())
print('OK | cwd =', os.getcwd())

In [ ]:
import yaml
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
QUICK  = True          # True = fast smoke test; False = full paper settings
print('device =', DEVICE, '| QUICK =', QUICK)

def show_figs(run_dir, names):
    fdir = os.path.join(run_dir, 'figures')
    for nm in names:
        p = os.path.join(fdir, nm)
        if os.path.exists(p):
            plt.figure(figsize=(7, 4.2)); plt.imshow(mpimg.imread(p)); plt.axis('off'); plt.show()
        else:
            print('(missing)', nm)

# QUICK shrinks ONLY the training budget (epochs / seeds / sweep length) so the
# notebook runs fast; it does NOT touch the algorithm knobs below.
def quicken(cfg, spatial=False):
    if not QUICK:
        return cfg
    if spatial:
        cfg.update(dsm_epochs=20, nmlp_epochs=10)
    else:
        cfg.update(seed=42, n_train_list=[100, 500], rho_list=[0.01, 0.1],
                   n_fixed_for_rho=200, dsm_epochs=50, lrao_epochs=20, test_size=400)
    return cfg

## Algorithm hyperparameters — edit these

These are the **detection-algorithm** knobs (they change *what each detector
computes*). Training settings — optimizer (`lr`, `weight_decay`, `batch_size`),
budget (`*_epochs`), early stopping, and network architecture — live in
`configs/*.yaml` and are intentionally **not** exposed here.

In [ ]:
# ---- IID (single-class and multi-class) ----
ALG_IID = dict(
    dsm_sigma_rho         = 0.1,      # DART / L-DART score-matching noise level rho
    whiten_eig_floor      = 0.0,      # ZCA eigenvalue floor for DART/L-DART (0 -> auto 1e-5*lambda_max)
    lrao_whiten_eig_floor = 1e-10,    # ZCA floor for the learned-Rao detectors
    lfi_delta_theta       = 0.01,     # LRao / L-LRao finite-difference step (Jacobian)
    lfi_sigma_cutoff      = 1e-6,     # LRao / L-LRao pseudo-inverse cutoff
    lfi_detach_sigma      = True,     # LRao / L-LRao: detach Sigma in the objective (stable)
)
ALG_IID_SINGLE = dict(ALG_IID, gmm_K=2)     # GMM-Levin components (single-class)
ALG_IID_MULTI  = dict(ALG_IID, gmm_K=9)     # GMM-Levin components (mixed background)

# ---- Spatial / correlated background ----
ALG_SPATIAL = dict(
    # DARTS neighbourhood (changing these retrains the spatial score net)
    k                    = 5,         # k x k spatial window
    nmlp_K               = 8,         # number of latent-nearest neighbours pooled
    # score-matching / whitening
    dsm_sigma_rho        = 0.01,
    whiten_eig_floor     = 1e-5,
    # CFAR normalisation (DART-CFAR / DARTS-CFAR)
    pfa_target           = 0.05,      # CFAR threshold target false-alarm rate
    cfar_lam             = 0.1,       # local -> global Fisher shrinkage (0 local, 1 global)
    cfar_fisher_use_topk = False,     # DARTS-CFAR Fisher set: False = all window nbrs, True = top-K
    sdsm_cfar_window     = None,      # DARTS-CFAR Fisher window (None = model's k)
    sdsm_cfar_guard      = 1,         # DARTS-CFAR guard block side
    dsm_cfar_window      = 11,        # DART-CFAR local window
    dsm_cfar_guard       = 3,         # DART-CFAR guard block side
    # classical baselines
    amf_local_window     = 15,        # AMF-local SCM window (>= D samples needed)
    local_scm_loading    = 0.0,       # AMF-local diagonal loading (0 = none)
    baseline_eig_floor   = 0.0,       # AMF covariance eigenvalue floor
    gmm_K                = 9,         # GMM-Levin components
    gmm_steps            = 50,        # GMM-Levin fill-factor grid steps
    # target / scene
    scenario_index       = 4,         # 0-3 manual boxes; 4+ random boxes
    foreign_class        = 7,         # planted foreign-target class (bitumen)
    amplitude            = 0.15,      # target amplitude theta
    target_fraction      = 0.10,
)
print('algorithm knobs ready')

## 1. IID — single-class background (Fig. 2)

In [ ]:
import src.iid as iid
cfg = yaml.safe_load(open('configs/iid_single.yaml'))
cfg['device'] = DEVICE
cfg.update(ALG_IID_SINGLE)
cfg = quicken(cfg)
run_dir_single, _ = iid.run_iid(cfg, mode='single')
show_figs(run_dir_single, ['pauc_vs_n.png', 'pd_at_fa_vs_n.png', 'pdet_at_pfa_vs_rho.png'])

## 2. IID — multi-class background (Fig. 3)

In [ ]:
cfg = yaml.safe_load(open('configs/iid_multi.yaml'))
cfg['device'] = DEVICE
cfg.update(ALG_IID_MULTI)
cfg = quicken(cfg)
run_dir_multi, _ = iid.run_iid(cfg, mode='multi')
show_figs(run_dir_multi, ['pauc_vs_n.png', 'pd_at_fa_vs_n.png', 'pdet_at_pfa_vs_rho.png'])

## 3. Spatial — correlated background (Table 1)

Trains the global DART and the spatially adapted DARTS on disjoint train/test
boxes, then compares all detectors and prints the summary table. With
`QUICK=False`, `run_multiseed` averages over seeds.

In [ ]:
import src.spatial as spatial
cfg = yaml.safe_load(open('configs/spatial.yaml'))
cfg['device'] = DEVICE
cfg.update(ALG_SPATIAL)
cfg = quicken(cfg, spatial=True)
if QUICK:
    parent = spatial.run_from_cfg(cfg, dry_run=False)         # one seed
    spatial.show_plots_from_dir(parent, sub='foreign', inline=True)
else:
    parent = spatial.run_multiseed(cfg, seeds=(42, 43, 44, 45, 46))
    spatial.show_multiseed(parent, inline=True)
print('spatial results ->', parent)

## CFAR ablation (optional)

Sweep `cfar_lam` (local→global Fisher shrinkage) for the DART-CFAR / DARTS-CFAR
rows of Table 1. This is a pure scoring knob — no retraining needed.

In [ ]:
for lam in (0.0, 0.5, 1.0):
    c = dict(cfg, cfar_lam=lam,
             active_detectors=['DART-CFAR', 'DARTS-CFAR', 'DARTS', 'AMF', 'GMM-Levin'])
    rd = spatial.run_from_cfg(c, dry_run=False)
    print(f'cfar_lam={lam} ->', rd)